# 08. 데이터·모델 개선과 기여 분리

개선 실험은 학습 셀 수를 4,096개로 맞춘 별도 조건이며, 전체 개발 평가 세포 63,036개를 유지합니다. A-series의 12,000개 학습 조건과 구분합니다. 기본 동작은 실제 저장 결과와 작업 상태를 읽는 것입니다. 새 학습은 마지막의 선택 옵션을 켰을 때만 Slurm에 제출합니다.


In [1]:
from pathlib import Path
import json, subprocess
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
REPORT = ROOT / 'reports/research_phase2'
protocol = json.loads((ROOT / 'configs/research/hypotheses_v1.json').read_text())
print(protocol['protocol'])


{'backbone': 'Geneformer-V2-104M base, not CLcancer', 'checkpoint_revision': '1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5', 'split_manifest': 'manifests/research_phase2/development_donor4_v2.tsv', 'evaluation_population': 'All 63036 development cells across 4 donor folds; legacy donors and external final excluded', 'training_budget': '4096 outer-training cells for all improvement controls; a separate matched training-budget condition from the 12000-cell A-series, never reuse A-series scores as the improvement baseline', 'validation_policy': 'same D0 capped inner-validation IDs within each seed/fold; data sampling modifies only training; final refit uses six outer-training donors', 'initial_screen': 'fold0 seed42 only; choose via inner-validation fixed-six macroF1, no external scores', 'confirmation': 'seeds42,43,44 on all4 donor folds for selected data/model candidates and paired controls', 'no_new_blind_claim': True}


## 데이터를 바꾼 첫 대조

D1은 donor-balanced, D2는 donor×class-balanced sampling입니다. 같은 학습 셀 수를 유지하고, validation/evaluation ID 및 C/class_weight를 고정했습니다. 두 변경은 첫 fold에서 모두 감소했습니다. D1은 후속 기여 분리를 위한 후보이며 확인된 개선이라는 뜻이 아닙니다.


In [2]:
display(pd.read_csv(REPORT / 'sampling_screen/metrics.tsv', sep='\t'))
print(json.loads((REPORT / 'sampling_screen/audit.json').read_text())['checks'])


,run_id,data_variant,n_outer_training,n_evaluation,inner_validation_macro_f1,development_outer_macro_f1,selected_C,selected_class_weight
0,D00_full4096_fold0_seed42_v1,D0_natural,4096,12082,0.938944,0.932355,0.01,balanced
1,D1_donor_balanced_full4096_fold0_seed42_v1,D1_donor_balanced,4096,12082,0.928110,0.916089,0.01,balanced
2,D2_donor_class_balanced_full4096_fold0_seed42_v1,D2_donor_class_balanced,4096,12082,0.918294,0.901284,0.01,balanced


['same seed/fold/checkpoint/head hyperparameters', 'identical training cell count', 'identical validation manifest hash', 'identical ordered evaluation population', 'no outer-training/evaluation overlap', 'actual manifest hashes verified']


## 같은 분류 head로 모델 변경 비교

M01은 동결 encoder + 768→128→6 NN head, M02는 사용되는 마지막 encoder block 9·10만 미세조정, M03은 block 0–10의 query/value 22개 모듈에 rank8 LoRA를 적용합니다. CLS pooling, V2 tokenizer/4096 제한, head 초기화와 optimizer 정책을 맞췄습니다. encoder dropout은 모두 껐으며 학습 parameter와 실제 update 수를 기록했습니다.

LoRA v1은 초기 adapter seed가 명시적으로 고정되지 않아 보존하되 선택에서 제외했습니다. 같은 hyperparameter의 수정 실행 v2를 사용합니다. 후보 선택은 첫 fold의 inner-validation 점수만 읽습니다.


In [3]:
rows = []
for name in ['M01_full4096_fold0_seed42_v1', 'M02_full4096_fold0_seed42_v1', 'M03_full4096_fold0_seed42_v2']:
    j = json.loads((REPORT / 'runs' / name / 'run.json').read_text())
    rows.append(dict(run_id=name, validation_F1=j['inner_validation_macro_f1'], development_outer_F1=j['metrics']['macro_f1'], encoder_parameters=j['trainable_encoder_parameters'], head_parameters=j['trainable_head_parameters'], epochs=j['chosen_epochs'], reload_max_difference=j['save_reload_probability_max_abs_difference']))
display(pd.DataFrame(rows))
selection = json.loads((ROOT / 'configs/research/confirmation_selection_v2.json').read_text())
print('frozen adapted candidate:', selection['selected_model'])
print('selection rule:', selection['selection_rule'])


,run_id,validation_F1,development_outer_F1,encoder_parameters,head_parameters,epochs,reload_max_difference
0,M01_full4096_fold0_seed42_v1,0.933992,0.931957,0,99206,1,0.0
1,M02_full4096_fold0_seed42_v1,0.947386,0.925856,14175744,99206,2,0.0
2,M03_full4096_fold0_seed42_v2,0.935934,0.928033,270336,99206,2,0.0


frozen adapted candidate: partial
selection rule: largest fold0 seed42 inner-validation fixed-six macroF1 among partial/LoRA; fixed-head frozen NN is the matched control


## 4개 fold·3개 seed의 동결 대조

M0는 위 M01과 같은 동결 NN head입니다. 각 seed에서 평가된 4개 donor fold의 예측을 합쳐 F1을 계산한 뒤 seed 평균·표준편차를 봅니다. CI는 독립 단위인 8명의 donor를 재표집하여 F1을 다시 계산합니다. seed를 독립 환자로 취급하지 않고, cell bootstrap도 사용하지 않습니다. 학습 fold가 겹치고 donor 수가 작다는 한계가 있습니다.


In [4]:
display(pd.read_csv(REPORT / 'factorial_frozen_controls/pooled_mean_std.tsv', sep='\t'))
display(pd.read_csv(REPORT / 'factorial_frozen_controls/paired_effects.tsv', sep='\t'))


,factor,mean,std
0,DM00,0.943991,0.003329
1,DM10,0.940069,0.008905


,effect,pooled_mean_difference,donor_bootstrap_lower95,donor_bootstrap_upper95,n_nonnegative_donors,n_donors
0,Data gain,-0.003922,-0.010603,0.002592,2,8


## 2×2 전체 결과와 실제 작업 상태

DM00=(D0,동결 NN), DM10=(D1,동결 NN), DM01=(D0,선택된 적응 모델), DM11=(D1,선택된 적응 모델)입니다. Interaction은 DM11−DM10−DM01+DM00으로 정의합니다. 전체 결과는 각 조건의 모든 4개 fold·3개 seed가 실제로 끝났을 때만 표시합니다.


In [5]:
path = REPORT / 'factorial_confirmation/paired_effects.tsv'
if path.exists():
    display(pd.read_csv(path, sep='\t'))
    display(pd.read_csv(REPORT/'factorial_confirmation/pooled_mean_std.tsv',sep='\t'))
    display(pd.read_csv(REPORT/'factorial_confirmation/per_class_mean_std.tsv',sep='\t'))
    display(pd.read_csv(REPORT/'factorial_confirmation/resources_by_factor.tsv',sep='\t'))
else:
    print('전체 적응 모델 확인이 아직 완료되지 않았습니다.')
job = REPORT / 'adapted_confirmation_slurm_job_id.txt'
if job.exists() and not (REPORT/'factorial_confirmation/provenance.json').exists():
    job_id = job.read_text().strip().split(';')[0]
    print('Slurm array:', job_id)
    result = subprocess.run(['squeue', '-j', job_id, '-o', '%.18i %.28j %.8T %.10M %R'], text=True, capture_output=True)
    print(result.stdout if result.returncode == 0 else result.stderr)


,effect,pooled_mean_difference,donor_bootstrap_lower95,donor_bootstrap_upper95,n_nonnegative_donors,n_donors
0,Data gain,-0.003922,-0.010603,0.002592,2,8
1,Model gain,0.001451,-0.002227,0.005839,4,8
2,Combined gain,-0.008685,-0.020099,0.001856,3,8
3,Interaction,-0.006215,-0.010518,-0.001306,1,8


,factor,mean,std
0,DM00,0.943991,0.003329
1,DM01,0.945443,0.001046
2,DM10,0.940069,0.008905
3,DM11,0.935306,0.004899


,factor,label,n_evaluation_support,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,DM00,B,2316,0.929427,0.007187,0.869603,0.001557,0.898513,0.003931
1,DM00,Endothelial,3433,0.928149,0.006915,0.946597,0.001682,0.937277,0.004299
2,DM00,Fibroblast,2042,0.945850,0.017934,0.931277,0.011509,0.938353,0.003010
3,DM00,Hepatocyte,16996,0.938916,0.005163,0.964462,0.022114,0.951396,0.008788
4,DM00,Myeloid,14672,0.972616,0.001881,0.973123,0.000905,0.972870,0.001390
5,DM00,T/NK,23577,0.973176,0.015189,0.958123,0.002397,0.965539,0.006453
6,DM10,B,2316,0.928879,0.033248,0.853483,0.024135,0.888978,0.002981
7,DM10,Endothelial,3433,0.926360,0.007561,0.936984,0.037819,0.931312,0.016878
8,DM10,Fibroblast,2042,0.924544,0.066291,0.929154,0.007481,0.925850,0.030669
9,DM10,Hepatocyte,16996,0.938652,0.000766,0.971032,0.007161,0.954558,0.003417


,factor,n_runs,mean_training_seconds,mean_inference_seconds,sum_current_process_wall_seconds,max_process_gpu_gib,median_process_gpu_gib,trainable_encoder_parameters,trainable_head_parameters
0,DM00,12,24.937191,4.859607,579.200697,0.304688,0.304688,0,99206
1,DM01,12,452.041821,505.791389,11761.749861,5.078125,4.320312,14175744,99206
2,DM10,12,28.678283,4.776016,731.446887,0.304688,0.304688,0,99206
3,DM11,12,453.343249,493.711944,11742.004345,5.179688,4.277344,14175744,99206


## 외부 cohort의 사전 고정 범위

GSE151530은25명·32개 HCC 표본의 알려진 다섯 비악성 세포 계열 전이 평가이며, 모델·데이터 선택을 고정한 뒤 최종 평가합니다. 원 저자 라벨에는 hepatocyte가 따로 없어서 malignant/unclassified를 Unknown으로 유지합니다. fixed-six 주 점수와 평가 가능한 다섯 lineage의 보조 점수 및 라벨 coverage를 함께 사전 정의했습니다. 이 보조 점수로 hepatocyte 전이를 검증했다고 주장하지 않습니다.


In [6]:
external = json.loads((REPORT / 'external_metadata/contract.json').read_text())
print('prepared cohort:', external['n_donors'], external['n_samples'], external['n_cells'])
print('label support:', external['label_support'])
print('preparation contract status:', external['status'])
print('primary metric:', external['primary_metric'])

freeze=REPORT/'external_final/freeze.json'
evaluation=REPORT/'external_final/evaluation.json'
print('Current external state:', 'evaluated fixed batch' if evaluation.exists() else 'frozen; inference/scoring not yet complete' if freeze.exists() else 'prepared; no release')


prepared cohort: 25 32 50023
label support: {'Unknown': 22811, 'T/NK': 17348, 'Myeloid': 4616, 'Endothelial': 2338, 'B': 1854, 'Fibroblast': 1056}
preparation contract status: prepared_unscored
primary metric: Fixed-six macroF1 on cells with known lineage ground truth, absent Hepatocyte class contributes0; report all supports and label coverage
Current external state: frozen; inference/scoring not yet complete


## 선택적인 한 fold 재실행

아래 옵션을 켜면 새 run ID로 동결 NN 조건 하나를 Slurm에 제출합니다. 제출 결과는 학습 완료와 다릅니다. 장기 학습은 서버 안내에 따라 Slurm에서 실행하며 기존 결과를 덮어쓰지 않습니다.


In [7]:
SUBMIT_ONE_FOLD = False
if SUBMIT_ONE_FOLD:
    from datetime import datetime, timezone
    run_id = 'tutorial08_frozen_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    result = subprocess.run(['sbatch', '--parsable', str(ROOT / 'slurm/improvement_single.sh'), '--model', 'frozen_nn', '--fold', '0', '--seed', '42', '--run-id', run_id], cwd=ROOT, check=True, text=True, capture_output=True)
    print('submitted job:', result.stdout.strip(), 'run:', run_id)
else:
    print('저장 결과 읽기 모드: 새 학습을 제출하지 않았습니다.')


저장 결과 읽기 모드: 새 학습을 제출하지 않았습니다.


## 세 번째 데이터 가설: KG 특징 압축

기존 pathway 집합의 중복 관측을 근거로 PCA32 압축을 고정하고4fold·seed42에서 실행했습니다. 매 fit의 gene z-score, pathway128 선택, scaler/PCA는 훈련 세포에서만 계산했습니다. 같은 gene membership degree·pathway size를 유지한 random prior도 같은32차원으로 압축했습니다. 이 추가 탐색은 이미 고정한 D1/partial confirmation 및 외부 평가 후보 선택에 사용하지 않습니다.


In [8]:
kg=json.loads((REPORT/'kg_compression_diagnostic/provenance.json').read_text())
print('actual heads:',kg['n_executed_heads'],'development cells:',kg['n_development_cells'],'seeds:',kg['n_seeds'])
print('preregistered point rule met:',kg['preregistered_point_rule_met'])
display(pd.read_csv(REPORT/'kg_compression_diagnostic/pooled_metrics.tsv',sep='	'))
display(pd.read_csv(REPORT/'kg_compression_diagnostic/paired_donor_effects.tsv',sep='	'))


actual heads: 20 development cells: 63036 seeds: 1
preregistered point rule met: False


,condition,n_evaluation,seed,pooled_macro_f1
0,embedding_only,63036,42,0.946825
1,real128,63036,42,0.946964
2,real32,63036,42,0.946593
3,random32,63036,42,0.947761
4,pathway32,63036,42,0.722845


,comparison,pooled_macro_f1_difference,lower95,upper95,n_nonnegative_donors,n_donors
0,Compression gain,-0.000371,-0.001117,0.000160,4,8
1,Real prior vs matched random,-0.001168,-0.002274,0.000631,2,8
2,Compressed prior vs embedding only,-0.000232,-0.001189,0.001242,4,8


In [9]:
SUBMIT_KG_ONE_FOLD = False
if SUBMIT_KG_ONE_FOLD:
    from datetime import datetime, timezone
    run_tag='tutorial08_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
    result=subprocess.run(['sbatch','--parsable',str(ROOT/'slurm/kg_compression_single.sh'),'--fold','0','--baseline-run','D00_full4096_fold0_seed42_v1','--run-tag',run_tag],cwd=ROOT,check=True,text=True,capture_output=True)
    print('submitted:',result.stdout.strip(),'tag:',run_tag)
else:
    print('저장 결과 읽기 모드: fixed4096-training 한 fold의 KG 재실행을 제출하지 않았습니다.')


저장 결과 읽기 모드: fixed4096-training 한 fold의 KG 재실행을 제출하지 않았습니다.
